# Stoic Qwen3 14B DPO Training with Unsloth (4-bit QLoRA)

**Phase 2:** Direct Preference Optimization on top of the Qwen3 14B SFT LoRA.

DPO sharpens persona behavior by contrasting preferred persona-voiced answers with generic, shallow, or out-of-voice answers.

**Base Model:** `unsloth/Qwen3-14B-unsloth-bnb-4bit`

**Starting Adapter:** `stoic_qwen3_14b_sft_unsloth_bnb_4bit/lora_adapters`

**Dataset:** v2 DPO preference pairs from the Stoic persona datagen pipeline

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Chat Template:** Qwen ChatML via `tokenizer.apply_chat_template`

## 1. Setup - Configuration, Environment, GPU Check

Set paths and DPO hyperparameters, install/update the training libraries, and verify the GPU runtime.

In [2]:
import os, sys, subprocess, importlib.util

from pathlib import Path

# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and

# torch.compile path before importing unsloth/transformers.

os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"

os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

# DGX Spark unified memory: force CUDA allocator reclamation instead of letting
# PyTorch treat all 128GB unified memory as endlessly cacheable CUDA memory.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

def _run(cmd):

    result = subprocess.run(cmd, text=True, capture_output=True)

    if result.returncode != 0:

        print(result.stdout[-1000:])

        print(result.stderr[-1000:])

        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

print("Installing/updating notebook dependencies...")

_run([sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall", "--no-cache-dir", "--no-deps", "unsloth", "unsloth_zoo"])

_run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])

_run([sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"])

# Pin PEFT to 0.18.0. peft main (0.19.dev) imports gptqmodel.AwqGEMMQuantLinear
# inside get_peft_model's AWQ dispatcher; that class was renamed to AwqGEMMLinear
# in gptqmodel 7.0, so get_peft_model raises ImportError on this container.
_run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft==0.18.0"])

import torch

if not torch.cuda.is_available():

    raise RuntimeError(f"No CUDA GPU available. torch={torch.__version__}")

_MEMORY_FRACTION = 0.55
try:
    torch.cuda.set_per_process_memory_fraction(_MEMORY_FRACTION, 0)
    _total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"CUDA memory cap: {_total_gb * _MEMORY_FRACTION:.1f} GB ({_MEMORY_FRACTION:.0%} of {_total_gb:.0f} GB) - allocator forced to reclaim")
except RuntimeError as exc:
    print(f"WARNING: could not set CUDA memory fraction: {exc}")
    print("Relying on PYTORCH_CUDA_ALLOC_CONF garbage collection only")
# =========================== PATHS ===========================

PROJECT_ROOT = Path("/workspace/training/stoic")

OUTPUT_ROOT = PROJECT_ROOT / "output"

# =========================== MODEL CONFIGURATION ===========================

BASE_LLM = "unsloth/Qwen3-14B-unsloth-bnb-4bit"

MODEL_NAME_BASE = "stoic_qwen3_14b_dpo_unsloth_bnb_4bit"

SFT_MODEL_NAME_BASE = "stoic_qwen3_14b_sft_unsloth_bnb_4bit"

SFT_LORA_PATH = OUTPUT_ROOT / SFT_MODEL_NAME_BASE / "lora_adapters"

# =========================== INPUT DATA ===========================

DPO_DATA_FILE = PROJECT_ROOT / "data/training-data/stoic_persona/stoic_personas_dpo.jsonl"

# =========================== OUTPUT DIRECTORIES ===========================

OUTPUT_BASE_DIR = OUTPUT_ROOT / MODEL_NAME_BASE

TRAIN_DIR = OUTPUT_BASE_DIR / "train"

LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / "lora_adapters"

# =========================== DPO HYPERPARAMETERS ===========================

MAX_SEQ_LENGTH = 4096

MAX_PROMPT_LENGTH = 2048

BATCH_SIZE = 1

GRAD_ACCUM = 8

LEARNING_RATE = 5e-6

TARGET_EPOCHS = 1

DPO_BETA = 0.1

LOSS_TYPE = "sigmoid"

WARMUP_RATIO = 0.1

SAVE_STEPS = 50

# =========================== INFERENCE TEST ===========================

TEST_PROMPT = "I am struggling with anxiety about things outside my control. How do I find peace?"

print("=" * 60)

print("ENVIRONMENT READY")

print("=" * 60)

print(f"torch:       {torch.__version__}")

print(f"CUDA:        {torch.version.cuda}")

print(f"GPU:         {torch.cuda.get_device_name(0)}")

print(f"Base model:  {BASE_LLM}")

print(f"SFT LoRA:    {SFT_LORA_PATH}")

print(f"DPO data:    {DPO_DATA_FILE}")

print(f"Output:      {OUTPUT_BASE_DIR}")

print(f"Training:    batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}, beta={DPO_BETA}")

Installing/updating notebook dependencies...
CUDA memory cap: 71.9 GB (55% of 131 GB) - allocator forced to reclaim
ENVIRONMENT READY
torch:       2.10.0a0+b558c986e8.nv25.11
CUDA:        13.0
GPU:         NVIDIA GB10
Base model:  unsloth/Qwen3-14B-unsloth-bnb-4bit
SFT LoRA:    /workspace/training/stoic/output/stoic_qwen3_14b_sft_unsloth_bnb_4bit/lora_adapters
DPO data:    /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_dpo.jsonl
Output:      /workspace/training/stoic/output/stoic_qwen3_14b_dpo_unsloth_bnb_4bit
Training:    batch=1, grad_accum=8, lr=5e-06, beta=0.1


## 2. Load DPO Dataset

Load the preference pairs produced by the Stoic datagen DPO notebook. Expected format: `{chosen: [...messages], rejected: [...messages], source: str, persona: str}`.

In [4]:
import json
from collections import Counter, defaultdict
from datasets import Dataset as HFDataset

if not DPO_DATA_FILE.exists():
    raise FileNotFoundError(f"DPO data not found: {DPO_DATA_FILE}")

with open(DPO_DATA_FILE) as f:
    raw_pairs = [json.loads(line) for line in f if line.strip()]

source_counts = Counter(pair.get("source", "unknown") for pair in raw_pairs)
persona_counts = Counter(pair.get("persona", "unknown") for pair in raw_pairs)
persona_system_prompts = {}

for pair in raw_pairs:
    persona = pair.get("persona", "unknown")
    chosen = pair.get("chosen", [])
    if chosen and chosen[0].get("role") == "system":
        persona_system_prompts.setdefault(persona, chosen[0].get("content", ""))

print(f"Loaded {len(raw_pairs)} DPO pairs from {DPO_DATA_FILE}")
print(f"Personas: {len(persona_counts)}")
print(f"System prompts extracted: {len(persona_system_prompts)}")

print("\nSource distribution:")
for source, count in source_counts.most_common():
    print(f"  {source:30s} {count:>6}")

print("\nTop personas:")
for persona, count in persona_counts.most_common(12):
    print(f"  {persona:30s} {count:>6}")

Loaded 1800 DPO pairs from /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_dpo.jsonl
Personas: 4
System prompts extracted: 4

Source distribution:
  shallow_platitude                 600
  voice_drift                       600
  source_fabrication                600

Top personas:
  epictetus                         600
  seneca                            600
  marcus_aurelius                   591
  epicurus                            9


## 3. Load Model & Tokenizer (4-bit)

Load the SFT LoRA as the DPO starting point. This continues training the same adapter instead of stacking a new LoRA.

In [ ]:
import os, tempfile, stat

# ── Neuter git BEFORE importing unsloth/transformers/peft/gptqmodel ──────────
# gptqmodel's startup banner (and a few other libs) shell out to `git rev-parse`
# to stamp a version. In the Unsloth container, /workspace/* is owned by the
# host user but git runs as root → "dubious ownership" failure, which trips up
# downstream code that wasn't expecting a non-zero git exit (most visibly
# hanging the LoRA setup / DPO trainer-init steps when peft pulls in gptqmodel).
# We don't use git for anything here, so prepend a fake `git` to PATH that
# always exits 0. Must run BEFORE any unsloth/transformers/peft imports.
_fake_git_dir = tempfile.mkdtemp(prefix="nogit-")
_fake_git = os.path.join(_fake_git_dir, "git")
with open(_fake_git, "w") as _f:
    _f.write("#!/bin/sh\nexit 0\n")
os.chmod(_fake_git, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
os.environ["PATH"] = _fake_git_dir + os.pathsep + os.environ.get("PATH", "")
print(f"✓ Fake git installed at {_fake_git} (PATH-shadows real git for this kernel)")

from unsloth import FastLanguageModel
import torch

if not SFT_LORA_PATH.exists():
    raise FileNotFoundError(
        f"SFT LoRA not found at {SFT_LORA_PATH}. Run the Qwen3 14B SFT notebook first."
    )

print(f"Loading SFT LoRA from: {SFT_LORA_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    str(SFT_LORA_PATH),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
model.config.pad_token_id = text_tokenizer.pad_token_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.pad_token_id = text_tokenizer.pad_token_id

print("Model + SFT LoRA loaded")
print(f"  Base:        {BASE_LLM}")
print(f"  Adapter:     {SFT_LORA_PATH}")
print(f"  Precision:   4-bit QLoRA")
print(f"  Max seq len: {MAX_SEQ_LENGTH}")
print(f"  Tokenizer:   {type(text_tokenizer).__name__}")
print(f"  GPU alloc:   {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. Validate & Prepare DPO Dataset

Convert conversational `chosen` / `rejected` examples into the `{prompt, chosen, rejected}` text format expected by TRL's `DPOTrainer`.

In [ ]:
errors = []
formatted_pairs = []
skipped_too_long = 0

for i, pair in enumerate(raw_pairs):
    chosen_msgs = pair.get("chosen", [])
    rejected_msgs = pair.get("rejected", [])

    if len(chosen_msgs) < 3 or len(rejected_msgs) < 3:
        errors.append(f"Pair {i}: too few messages (chosen={len(chosen_msgs)}, rejected={len(rejected_msgs)})")
        continue

    if chosen_msgs[:-1] != rejected_msgs[:-1]:
        errors.append(f"Pair {i}: chosen/rejected prompt mismatch")
        continue

    if chosen_msgs[-1].get("role") != "assistant" or rejected_msgs[-1].get("role") != "assistant":
        errors.append(f"Pair {i}: final messages must both be assistant responses")
        continue

    prompt_msgs = chosen_msgs[:-1]
    chosen_response = chosen_msgs[-1].get("content", "")
    rejected_response = rejected_msgs[-1].get("content", "")

    if not chosen_response.strip() or not rejected_response.strip():
        errors.append(f"Pair {i}: empty chosen or rejected response")
        continue
    if chosen_response.strip() == rejected_response.strip():
        errors.append(f"Pair {i}: chosen and rejected responses are identical")
        continue

    prompt_kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, enable_thinking=False, **prompt_kwargs)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, **prompt_kwargs)

    p_len = len(text_tokenizer.encode(prompt_text, add_special_tokens=False))
    c_len = len(text_tokenizer.encode(chosen_response, add_special_tokens=False))
    r_len = len(text_tokenizer.encode(rejected_response, add_special_tokens=False))

    if p_len > MAX_PROMPT_LENGTH or p_len + max(c_len, r_len) > MAX_SEQ_LENGTH:
        skipped_too_long += 1
        continue

    formatted_pairs.append({
        "prompt": prompt_text,
        "chosen": chosen_response,
        "rejected": rejected_response,
    })

if errors:
    print(f"WARNING: {len(errors)} validation errors")
    for error in errors[:10]:
        print(f"  {error}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more")
else:
    print(f"All {len(raw_pairs)} pairs passed validation")

if skipped_too_long:
    print(f"Filtered {skipped_too_long} pairs exceeding prompt/sequence limits")

dpo_dataset = HFDataset.from_list(formatted_pairs).shuffle(seed=42)

prompt_lens = [len(p["prompt"]) for p in formatted_pairs]
chosen_lens = [len(p["chosen"]) for p in formatted_pairs]
rejected_lens = [len(p["rejected"]) for p in formatted_pairs]

print(f"\nFormatted DPO dataset: {len(dpo_dataset)} pairs")
print(f"Columns: {dpo_dataset.column_names}")
print("\nLength stats (chars):")
print(f"  Prompt:   avg={sum(prompt_lens)//len(prompt_lens):,}, max={max(prompt_lens):,}")
print(f"  Chosen:   avg={sum(chosen_lens)//len(chosen_lens):,}, max={max(chosen_lens):,}")
print(f"  Rejected: avg={sum(rejected_lens)//len(rejected_lens):,}, max={max(rejected_lens):,}")

sample = formatted_pairs[0]
print("\nSample formatted pair:")
print(f"PROMPT:   {sample['prompt'][:300]}...")
print(f"CHOSEN:   {sample['chosen'][:200]}...")
print(f"REJECTED: {sample['rejected'][:200]}...")

del raw_pairs

## 5. Prepare SFT LoRA for DPO Training

Continue training the loaded SFT adapter directly. The output remains a single LoRA relative to the Qwen3 14B base model.

In [ ]:
FastLanguageModel.for_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print("SFT LoRA adapter ready for DPO training")
print(f"  Trainable params: {trainable:,}")
print(f"  Total params:     {total:,}")
print(f"  Trainable pct:    {pct:.4f}%")

## 6. DPO Trainer Setup

Configure TRL's `DPOTrainer` with a low learning rate and an implicit Unsloth reference model.

In [ ]:
from trl import DPOTrainer, DPOConfig
import math

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(dpo_dataset) / effective_batch)
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

print("DPO Training Plan")
print(f"  DPO pairs:       {len(dpo_dataset)}")
print(f"  Effective batch: {effective_batch}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total steps:     {max_steps}")
print(f"  Warmup steps:    {warmup_steps}")
print(f"  DPO beta:        {DPO_BETA}")

TRAIN_DIR.mkdir(parents=True, exist_ok=True)

if not hasattr(model, "warnings_issued"):
    object.__setattr__(model, "warnings_issued", {})
if hasattr(model, "base_model"):
    if not hasattr(model.base_model, "warnings_issued"):
        object.__setattr__(model.base_model, "warnings_issued", {})
    if hasattr(model.base_model, "model") and not hasattr(model.base_model.model, "warnings_issued"):
        model.base_model.model.warnings_issued = {}

# Qwen3 14B is text-only, so TRL should pick the text path without help.
# Keep a defensive swap in case a future config surfaces a nested text_config:
# TRL inspects model_type to route between text and vision pipelines.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
_orig_model_type = getattr(model.config, "model_type", None)
_text_model_type = None
if hasattr(model.config, "text_config") and model.config.text_config is not None:
    _text_model_type = getattr(model.config.text_config, "model_type", None)

if _text_model_type and _text_model_type != _orig_model_type:
    model.config.model_type = _text_model_type
    print(f"  Temporarily swapped model.config.model_type: {_orig_model_type!r} -> {_text_model_type!r}")
else:
    print(f"  model.config.model_type ({_orig_model_type!r}) left unchanged")

print(f"  processing_class: {type(text_tokenizer).__name__}")

try:
    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        processing_class=text_tokenizer,
        train_dataset=dpo_dataset,
        args=DPOConfig(
            beta=DPO_BETA,
            loss_type=LOSS_TYPE,
            max_length=MAX_SEQ_LENGTH,
            max_prompt_length=MAX_PROMPT_LENGTH,
            # NOTE: precompute is False here on purpose — the next cell does it
            # with a persistent, resumable shard cache so a crash mid-precompute
            # doesn't force starting over from row 0.
            precompute_ref_log_probs=False,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=warmup_steps,
            max_steps=max_steps,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            optim="adamw_8bit",
            weight_decay=0.0,
            seed=3407,
            gradient_checkpointing=True,
            dataloader_pin_memory=False,
            output_dir=str(TRAIN_DIR),
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=3,
            logging_steps=5,
            report_to="none",
            dataset_num_proc=1,
        ),
    )
finally:
    if _orig_model_type is not None:
        model.config.model_type = _orig_model_type

trainer.is_vision_model = False

print("\nDPO Trainer configured")
print(f"  trainer.is_vision_model: {trainer.is_vision_model}")
print(f"  model.config.model_type: {model.config.model_type!r}")
print("  precompute_ref_log_probs: False (persistent cache cell handles this)")
print("  dataloader_pin_memory: False")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

## 7. Precompute Reference Log Probabilities (persistent, resumable cache)

DPO needs the frozen reference model's log probabilities for every (prompt, chosen) and (prompt, rejected) in the dataset. TRL's built-in precompute is one-shot and in-memory — if the kernel dies halfway through (OOM, lost SSH, transient CUDA error), the next run starts from row 0.

This cell instead:
- Splits the dataset into shards of 64 rows
- Saves each completed shard to `TRAIN_DIR/ref_logprobs_cache/shards/`
- Writes a fingerprinted manifest so we can detect stale caches (different base model, SFT LoRA, data file, or sequence lengths invalidate and rebuild)
- Skips shards that already exist on disk → a failed run resumes from the first missing shard
- Concatenates all shards into a single dataset with `ref_chosen_logps` / `ref_rejected_logps` columns and hands it back to the trainer

Running this BEFORE training is intentional — once the cache is built, the training cell is pure gradient work and doesn't need the reference model loaded for forward passes.

In [ ]:
# Persistent, resumable reference logprob cache.
# TRL's built-in precompute is one-shot and in-memory; this saves completed
# shards so a failed run resumes from the first missing shard instead of zero.
import hashlib
import json
import os
import time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

_ref_cache_config = {
    "cache_version": 1,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns if DPO_DATA_FILE.exists() else None,
    "dataset_len": len(trainer.train_dataset),
    "dataset_columns": sorted([c for c in trainer.train_dataset.column_names if not c.startswith("ref_")]),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "tokenizer_class": type(text_tokenizer).__name__,
    "tokenizer_vocab_size": len(text_tokenizer),
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()

def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    with open(REF_LOGPROBS_MANIFEST) as f:
        return json.load(f)


def _load_valid_ref_shard(shard_path, shard_idx, start, end):
    try:
        shard_payload = torch.load(shard_path, map_location="cpu")
    except (EOFError, OSError, RuntimeError, ValueError) as exc:
        raise RuntimeError(f"Unreadable ref logprob shard: {shard_path}") from exc

    if not isinstance(shard_payload, dict):
        raise RuntimeError(f"Invalid ref logprob shard payload: {shard_path}")
    if shard_payload.get("fingerprint") != _ref_cache_fingerprint:
        raise RuntimeError(f"Stale ref logprob shard fingerprint: {shard_path}")
    if shard_payload.get("shard_idx") != shard_idx:
        raise RuntimeError(f"Ref logprob shard index mismatch: {shard_path}")
    if shard_payload.get("start") != start or shard_payload.get("end") != end:
        raise RuntimeError(f"Ref logprob shard range mismatch: {shard_path}")

    expected_rows = end - start
    for column in ("ref_chosen_logps", "ref_rejected_logps"):
        values = shard_payload.get(column)
        if not isinstance(values, torch.Tensor) or values.numel() != expected_rows:
            raise RuntimeError(f"Invalid {column} values in ref logprob shard: {shard_path}")
    return shard_payload


def _write_ref_manifest(status, completed_shards):
    manifest = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)

_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint
_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}

if _cache_matches and REF_LOGPROBS_DATASET_DIR.exists():
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    if len(cached_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(cached_dataset.column_names):
        trainer.train_dataset = cached_dataset
        trainer._precomputed_train_ref_log_probs = True
        print(f"Loaded persistent ref logprob cache: {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: length or columns do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        for stale_shard in REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"):
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])
        _manifest = _read_ref_manifest()

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]

    print("Persistent reference logprob cache")
    print(f"  Rows:       {num_rows}")
    print(f"  Shards:     {len(shard_ranges)}")
    print(f"  Shard size: {REF_LOGPROBS_SHARD_SIZE}")
    print(f"  Cache dir:  {REF_LOGPROBS_CACHE_DIR}")

    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            try:
                _load_valid_ref_shard(shard_path, shard_idx, start, end)
            except RuntimeError as exc:
                print(f"Recomputing corrupt ref logprob shard {shard_idx}: {exc}")
                shard_path.unlink()
            else:
                completed.append(shard_idx)
                continue

        shard_dataset = trainer.train_dataset.select(range(start, end))
        shard_loader = DataLoader(
            shard_dataset,
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps = []
        ref_rejected_logps = []
        for padded_batch in tqdm(shard_loader, desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}"):
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(padded_batch)
            ref_chosen_logp, ref_rejected_logp = trainer.accelerator.gather_for_metrics(
                (ref_chosen_logp, ref_rejected_logp)
            )
            ref_chosen_logps.append(ref_chosen_logp.float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.float().cpu())
            torch.cuda.empty_cache()
            trainer.accelerator.free_memory()

        shard_payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(shard_payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

    all_ref_chosen_logps = []
    all_ref_rejected_logps = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if not shard_path.exists():
            raise RuntimeError(f"Missing ref logprob shard: {shard_path}")
        shard_payload = _load_valid_ref_shard(shard_path, shard_idx, start, end)
        all_ref_chosen_logps.append(shard_payload["ref_chosen_logps"])
        all_ref_rejected_logps.append(shard_payload["ref_rejected_logps"])

    ref_chosen_values = torch.cat(all_ref_chosen_logps).numpy()
    ref_rejected_values = torch.cat(all_ref_rejected_logps).numpy()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError("Ref logprob cache length does not match training dataset")

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ["ref_chosen_logps", "ref_rejected_logps"]:
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        import shutil
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    trainer._precomputed_train_ref_log_probs = True
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

print(f"Ref logprob columns ready: {sorted(_required_ref_cols)}")


## 8. Train

DPO loss typically starts around log(2), about 0.69. Reward margins should widen as chosen responses become preferred over rejected responses.

Because step 7 already cached `ref_chosen_logps` / `ref_rejected_logps` into the training dataset, this cell does pure policy gradient work — the reference model is not consulted again.

In [ ]:
import gc
from transformers import TrainerCallback


class CudaCacheClearCallback(TrainerCallback):
    """Flush CUDA cache before and after every training step on unified memory."""

    def on_step_begin(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()

    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()


trainer.add_callback(CudaCacheClearCallback())

required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
if not required_ref_cols.issubset(trainer.train_dataset.column_names):
    raise RuntimeError("Run the persistent reference-logprob cache cell before training.")

from transformers.trainer_utils import get_last_checkpoint

print("DPO Training started...")
print("Watch for loss to decrease from about 0.69 toward 0.40-0.55.")
print("Reward margins should widen (chosen > rejected).\n")

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    trainer.train()

print("\nDPO training complete")


## 9. Save DPO LoRA Adapter

Save the combined SFT+DPO LoRA adapter. Load this adapter directly on top of `unsloth/Qwen3-14B-unsloth-bnb-4bit` / compatible Qwen3 14B base weights.

In [ ]:
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

prompts_path = LORA_OUTPUT_DIR / "persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

print(f"DPO LoRA adapter saved to: {LORA_OUTPUT_DIR}")
print(f"Persona prompts saved to: {prompts_path}")

total_size = 0
for path in sorted(LORA_OUTPUT_DIR.iterdir()):
    size = path.stat().st_size
    total_size += size
    print(f"  {path.name:40s} {size / 1024 / 1024:8.1f} MB")
print(f"  {'TOTAL':40s} {total_size / 1024 / 1024:8.1f} MB")

## 10. Quick Evaluation

Smoke test a few personas using their extracted system prompts. The DPO model should keep the persona voice while avoiding generic answers.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_personas = list(persona_system_prompts.keys())[:4]
print(f"Quick evaluation across {len(test_personas)} personas\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = text_tokenizer(text, return_tensors="pt").to(model.device)

    print("=" * 70)
    print(f"PERSONA: {persona_key.upper()}")
    print(f"Q: {TEST_PROMPT}")
    print("A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(text_tokenizer, skip_prompt=True),
    )
    print("\n")

del inputs, outputs

## 11. Verify Adapter Reload

Reload the saved DPO adapter from disk to confirm it is self-contained and portable.

In [ ]:
import gc

del model, tokenizer, trainer, dpo_dataset
gc.collect()
torch.cuda.empty_cache()

print(f"Reloading adapter from: {LORA_OUTPUT_DIR}")
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

text_tokenizer2 = getattr(tokenizer2, "tokenizer", tokenizer2)

with open(LORA_OUTPUT_DIR / "persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_key = list(reloaded_prompts.keys())[0]
messages = [
    {"role": "system", "content": reloaded_prompts[test_key]},
    {"role": "user", "content": TEST_PROMPT},
]

try:
    text = tokenizer2.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError:
    text = tokenizer2.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = text_tokenizer2(text, return_tensors="pt").to(model2.device)
outputs = model2.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)
response = text_tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"Adapter reload test persona: {test_key}")
print(f"Q: {TEST_PROMPT}")
print(f"A: {response[:500]}")
print("\nAdapter reload test complete")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

## Export Merged Model to GGUF (Mobile / Cross-Platform)

Merge the SFT+DPO LoRA adapter into the base Qwen3 14B and export to GGUF
so it can run anywhere llama.cpp / Ollama / LM Studio / iOS GGUF runners
are supported (e.g. iPhone via apps like LLMFarm, PocketPal, Private LLM).

Uses Unsloth's `save_pretrained_gguf` — same pattern as the biblical
gemma e2b export. Unsloth manages its own bundled `llama.cpp` build
internally; the cell does not depend on any external llama.cpp checkout.

**Output layout:**
- `<base>/gguf/` — intermediate FP16 HF merge
- `<base>/gguf_gguf/` — final `.gguf` quant files

**Quants produced:** `q2_k`, `q3_k_m`, `q4_k_s`, `q4_k_m`, `q5_k_s`, `q5_k_m`, `q6_k`, `q8_0`. Adjust `QUANT_METHODS` in the cell to change the set.

In [ ]:
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model, tokenizer, trainer
except NameError:
    pass
try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter will merge before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Force the chat-template's end-of-turn token as the GGUF EOS.
#
# Qwen ChatML uses `<|im_end|>` as the closer of `<|im_start|>role\n...content...<|im_end|>\n`.
# Without this fix, llama.cpp's GGUF converter may inherit a different EOS
# (e.g. `<|endoftext|>`), so the model never stops on `<|im_end|>` and emits
# the next turn header. iOS GGUF runners then strip the leading `<|im_start|>`
# special token and render the bare role text ("user\n...") plus a
# hallucinated user question.
_render = tokenizer_gguf.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False,
)
_eot_candidates = ["<|im_end|>", "<turn|>", "<end_of_turn>", "<|eot_id|>"]
eot_token = next((c for c in _eot_candidates if c in _render), None)
if eot_token is None:
    raise RuntimeError(f"Could not detect end-of-turn marker in chat template. Rendered: {_render!r}")
# Some loaders wrap the tokenizer; unwrap so we can call tokenizer methods.
_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
eot_id = _tok.convert_tokens_to_ids(eot_token)
if eot_id is None or eot_id == _tok.unk_token_id:
    raise RuntimeError(f"{eot_token!r} not in tokenizer vocab (got id={eot_id})")

_tok.eos_token = eot_token
model_gguf.config.eos_token_id = eot_id
# Defensive: some HF configs nest a text_config; if present, set there too.
if hasattr(model_gguf.config, "text_config") and model_gguf.config.text_config is not None:
    model_gguf.config.text_config.eos_token_id = eot_id
# generation_config.eos_token_id may be a list; collapse to the single
# chat-template EOS so runners that only honor a scalar pick the right one.
if getattr(model_gguf, "generation_config", None) is not None:
    model_gguf.generation_config.eos_token_id = eot_id

print(f"✓ Forced GGUF EOS to {eot_token!r} (id={eot_id})")

# Pass ALL quant methods in one call so the LoRA→FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q2_k", "q3_k_m", "q4_k_s", "q4_k_m", "q5_k_s", "q5_k_m", "q6_k", "q8_0"]

print(f"Exporting GGUF (single merge → {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\n✓ GGUF export complete: {GGUF_OUTPUT_DIR}")
# Unsloth writes quants into <GGUF_OUTPUT_DIR>_gguf alongside the merge dir.
for d in (Path(GGUF_OUTPUT_DIR), Path(f"{GGUF_OUTPUT_DIR}_gguf")):
    if not d.exists():
        continue
    for f in sorted(d.glob("*.gguf")):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name:60s} {size_mb:>8.1f} MB  ({d.name}/)")

print("\nMobile usage:")
print("  1. Pick a quant from <base>/gguf_gguf/  (q4_k_m is the standard phone quant).")
print("  2. Open in any llama.cpp / Ollama / LM Studio / iOS GGUF runner.")
print("  3. Use Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.5.7: Fast Qwen3 patching. Transformers: 5.10.0.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

unsloth/qwen3-14b does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.



/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 6/6 [04:24<00:00, 44.06s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 6/6 [02:45<00:00, 27.60s/it]


Unsloth: Merge process complete. Saved to `/workspace/training/stoic/output/stoic_qwen3_14b_dpo_unsloth_bnb_4bit/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q2_k', 'q3_k_m', 'q4_k_s', 'q4_k_m', 'q5_k_s', 'q5_k_m', 'q6_k', 'q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|ERROR]Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_rbr1oadl from /root/.unsloth/llama.cpp/original_gguf_rbr1oadl.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 914, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_rbr1oadl.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 1178, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_fil